# 06 · Aim 3: varenicline recipients, SVE/LVE, and mediation by cigarettes per day

Among all participants with a varenicline prescription (regardless of concurrent NRT):
- **All recipients:** first survey response after the first varenicline prescription
- **SVE (short-term):** first response 12–104 weeks after the index prescription
- **LVE (long-term):** first response ≥ 52 weeks after the index prescription

The PGS → cessation association is decomposed into an indirect path through self-reported cigarettes per day and a direct path (lavaan).
As a sensitivity analysis, smoking years and starting age are added as further mediators.



In [ ]:
# Setup: run from the repository root inside the All of Us Researcher Workbench
install_if_missing(c("tidyverse", "bigrquery", "allofus", "openxlsx", "lavaan"))
con <- aou_connect()

## 1. Varenicline recipients and smoking-survey responses

In [ ]:
cb_ancestor <- tbl(con, "cb_criteria_ancestor")
drug_tbl    <- tbl(con, "ds_drug_exposure")

# Varenicline and all descendant drug concepts
varenicline <- cb_ancestor %>%
  filter(ancestor_id == 780442) %>%
  select(descendant_id) %>%
  pull()

treat_vare <- drug_tbl %>%
  filter(DRUG_CONCEPT_ID %in% varenicline) %>%
  collect()

# NOTE (cleanup): this group is all varenicline recipients in All of Us. It is
# not restricted to the smoking-history cohort or the study window used in
# Aims 1–2. Check that this matches the paper's Methods.
cohort <- treat_vare %>%
  distinct(PERSON_ID) %>%
  rename(person_id = PERSON_ID)

survey <- aou_survey(
  cohort,
  questions = c(1585864, 1585867, 1585860, 1585873, 1585857, 1585870, 1586159),
  collect = TRUE
)

In [ ]:
############################################################
survey_outcomes <- survey %>%
  select(person_id, 
         Smoking_SmokeFrequency_date,
         Smoking_SmokeFrequency,
         Smoking_CurrentDailyCigaretteNumber,
         Smoking_CurrentDailyCigaretteNumber_date,
         Smoking_SeriousQuitAttempt,
         Smoking_SeriousQuitAttempt_date,
         Smoking_DailySmokeStartingAge,
         Smoking_100CigsLifetime,
         Smoking_NumberOfYears,
         AttemptQuitSmoking_CompletelyQuitAge
         ) %>%
  mutate(
    smoking_status = case_when(
      Smoking_SmokeFrequency == "EveryDay" | Smoking_SmokeFrequency == "SomeDays" ~ "current_smoker",
      Smoking_SmokeFrequency == "NotAtAll" ~ "quit_success",
      TRUE ~ "unknown"
    ),
    quit_attempt = case_when(
      Smoking_SeriousQuitAttempt == "AttemptQuitSmoking" ~ "yes",
      Smoking_SeriousQuitAttempt == "NoAttemptQuitSmoking" ~ "no",
      TRUE ~ "unknown"
    )
  )

index_dates <- treat_vare %>%
  group_by(PERSON_ID) %>%
  summarize(index_date = min(DRUG_EXPOSURE_START_DATETIME)) %>%
  ungroup()

## 2. Outcome windows: all recipients, SVE, LVE

In [ ]:
vare_all <- survey_outcomes %>%
  inner_join(index_dates, by = c("person_id" = "PERSON_ID")) %>%
  mutate(
    weeks_from_index = as.numeric(difftime(Smoking_SmokeFrequency_date, index_date, units = "weeks"))
  )%>%
  filter(weeks_from_index > 0) %>%
  group_by(person_id) %>%
  slice(1) %>%  # Take first record per person
  ungroup()

In [ ]:
# Join and calculate weeks from index
outcome_data <- survey_outcomes %>%
  inner_join(index_dates, by = c("person_id" = "PERSON_ID")) %>%
  mutate(
    weeks_from_index = as.numeric(difftime(Smoking_SmokeFrequency_date, index_date, units = "weeks"))
  )

# SVE: First record between 12-104 weeks
sve_outcomes <- outcome_data %>%
  filter(weeks_from_index >= 12 & weeks_from_index <= 104) %>%
  arrange(person_id, weeks_from_index) %>%
  group_by(person_id) %>%
  slice(1) %>%  # Take first record per person
  ungroup() %>%
  mutate(
    phenotype = "SVE",
    case_control = if_else(smoking_status == "quit_success", "case", "control")
  )

# LVE: First record after 52 weeks (≥52 weeks)
lve_outcomes <- outcome_data %>%
  filter(weeks_from_index >= 52) %>%
  arrange(person_id, weeks_from_index) %>%
  group_by(person_id) %>%
  slice(1) %>%  # Take first record per person
  ungroup() %>%
  mutate(
    phenotype = "LVE",
    case_control = if_else(smoking_status == "quit_success", "case", "control")
  )

# Check overlap: people appearing in both phenotypes
overlap_check <- sve_outcomes %>%
  inner_join(
    lve_outcomes %>% select(person_id, weeks_from_index_lve = weeks_from_index),
    by = "person_id"
  ) %>%
  mutate(
    same_record = (weeks_from_index == weeks_from_index_lve)
  )

# Summary statistics
cat("SVE sample size:", nrow(sve_outcomes), "\n")
cat("  Cases (quit):", sum(sve_outcomes$case_control == "case"), "\n")
cat("  Controls (smoking):", sum(sve_outcomes$case_control == "control"), "\n\n")

cat("LVE sample size:", nrow(lve_outcomes), "\n")
cat("  Cases (quit):", sum(lve_outcomes$case_control == "case"), "\n")
cat("  Controls (smoking):", sum(lve_outcomes$case_control == "control"), "\n\n")

cat("Overlap:", nrow(overlap_check), "people in both\n")
cat("  Using same record:", sum(overlap_check$same_record), "\n")
cat("  Using different records:", sum(!overlap_check$same_record), "\n")

## 3. Add covariates

The original notebook repeated the same join block three times (for all recipients, SVE and LVE); here it is a single function.

In [ ]:
person_covariates <- load_from_bucket("person_covariates.csv")

add_covariates <- function(df) {
  df %>%
    mutate(person_id = as.double(person_id)) %>%
    left_join(person_covariates, by = "person_id") %>%
    mutate(
      # Age at first varenicline prescription
      age_at_treatment = as.numeric(difftime(index_date, birth_date, units = "days")) / 365.25,
      Smoking_Years_Numeric = case_when(
        Smoking_NumberOfYears %in% c("DontKnow", "PreferNotToAnswer", "Skip") ~ NA_real_,
        !is.na(Smoking_NumberOfYears) ~ suppressWarnings(as.numeric(as.character(Smoking_NumberOfYears))),
        TRUE ~ NA_real_
      ),
      Cigarettes_Per_Day_Numeric = suppressWarnings(as.numeric(as.character(Smoking_CurrentDailyCigaretteNumber))),
      Starting_Age_Numeric       = suppressWarnings(as.numeric(as.character(Smoking_DailySmokeStartingAge))),
      Smoking_100CigsLifetime_Binary = case_when(
        Smoking_100CigsLifetime == "Yes" ~ 1,
        Smoking_100CigsLifetime == "No"  ~ 0,
        TRUE ~ NA_real_
      )
    ) %>%
    # Cessation outcome; "unknown" responses are dropped
    filter(smoking_status != "unknown") %>%
    mutate(quit_success = ifelse(smoking_status == "current_smoker", 0, 1))
}

vare_all     <- add_covariates(vare_all)
sve_outcomes <- add_covariates(sve_outcomes)
lve_outcomes <- add_covariates(lve_outcomes)

save_to_bucket(vare_all,     "vare_all.csv")
save_to_bucket(sve_outcomes, "sve_outcomes.csv")
save_to_bucket(lve_outcomes, "lve_outcomes.csv")

## 4. Descriptives (Table 1, varenicline columns)

In [ ]:
describe_cohort(vare_all,     age_var = "age_at_treatment")
describe_cohort(sve_outcomes, age_var = "age_at_treatment")
describe_cohort(lve_outcomes, age_var = "age_at_treatment")
# NOTE (cleanup): the original SVE/LVE quit-rate tables divided by nrow(vare_all);
# describe_cohort() uses each group's own N.

In [ ]:
library(dplyr)

# 1. Identify the overlapping individuals
overlap_ids <- intersect(sve_outcomes$person_id, lve_outcomes$person_id)
overlap_df <- vare_all %>% filter(person_id %in% overlap_ids)

# 2. Calculate the "XXX" counts and percentages
n_overlap <- length(overlap_ids)
pct_overlap <- (n_overlap / n_distinct(vare_all$person_id)) * 100

# 3. Calculate Demographics for the overlap group
# Assuming 'sex' is the column for Female/Male and 'age_at_treatment' is the age
stats_overlap <- overlap_df %>%
  summarise(
    n = n(),
    pct_female = mean(sex == "Female", na.rm = TRUE) * 100, # Adjust string if it's "F" or a code
    mean_age = mean(age_at_treatment, na.rm = TRUE),
    sd_age = sd(age_at_treatment, na.rm = TRUE)
  )

# Print results to fill in your text
print(stats_overlap)

## 5. Logistic models: PGS → cessation among varenicline recipients

Model 1: PGS + demographics + PCs · Model 2: + psychiatric · Model 3: + psychiatric + SDoH. Exported as one workbook with GVE/SVE/LVE tabs.

In [ ]:
# ── Master model fitting function ─────────────────────────────────────────────
fit_effectiveness_models <- function(data, prs_var) {
  
  # Model 1: Base (PRS + Demographics)
  m1_EUR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10")),
    data = data %>% filter(ancestry == "EUR"), family = binomial())
  
  m1_AFR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10")),
    data = data %>% filter(ancestry == "AFR"), family = binomial())
  
  # Model 2: + Psychiatric
  m2_EUR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10+
    SCZ + ADHD + Bipolar_Disorder + Anxiety + Depression + PTSD")),
    data = data %>% filter(ancestry == "EUR"), family = binomial())
  
  m2_AFR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10+
    SCZ + ADHD + Bipolar_Disorder + Anxiety + Depression + PTSD")),
    data = data %>% filter(ancestry == "AFR"), family = binomial())
  
  # Model 3: + Psychiatric + SDoH
  m3_EUR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10+
    SCZ + ADHD + Bipolar_Disorder + Anxiety + Depression + PTSD +
    DEPRIVATION_INDEX + edu + income + insurance_cat + employment_status")),
    data = data %>% filter(ancestry == "EUR"), family = binomial())
  
  m3_AFR <- glm(as.formula(paste0(
    "quit_success ~", prs_var, "+ sex + age_at_treatment +
    PC1+PC2+PC3+PC4+PC5+PC6+PC7+PC8+PC9+PC10+
    SCZ + ADHD + Bipolar_Disorder + Anxiety + Depression + PTSD +
    DEPRIVATION_INDEX + edu + income + insurance_cat + employment_status")),
    data = data %>% filter(ancestry == "AFR"), family = binomial())
  
  # Return all 6 models as a named list
  list(m1_EUR = m1_EUR, m1_AFR = m1_AFR,
       m2_EUR = m2_EUR, m2_AFR = m2_AFR,
       m3_EUR = m3_EUR, m3_AFR = m3_AFR)
}

# ── Fit all models ─────────────────────────────────────────────────
gve_CPD <- fit_effectiveness_models(vare_all,    "PRS_CPD_scaled")
gve_CES <- fit_effectiveness_models(vare_all,    "PRS_CES_scaled")
sve_CPD <- fit_effectiveness_models(sve_outcomes,"PRS_CPD_scaled")
sve_CES <- fit_effectiveness_models(sve_outcomes,"PRS_CES_scaled")
lve_CPD <- fit_effectiveness_models(lve_outcomes,"PRS_CPD_scaled")
lve_CES <- fit_effectiveness_models(lve_outcomes,"PRS_CES_scaled")

# ── Extract results ───────────────────────────────────────────────────────────
extract_model_aim3 <- function(model, model_name, ancestry, prs_type, outcome_group) {
  coefs <- summary(model)$coefficients
  or_ci <- exp(cbind(OR = coef(model), confint(model)))
  
  data.frame(
    outcome_group = outcome_group,
    model         = model_name,
    ancestry      = ancestry,
    prs_type      = prs_type,
    predictor     = rownames(coefs),
    OR            = round(or_ci[, "OR"], 3),
    CI_lower      = round(or_ci[, "2.5 %"], 3),
    CI_upper      = round(or_ci[, "97.5 %"], 3),
    p_value       = round(coefs[, "Pr(>|z|)"], 4),
    n             = nobs(model),
    stringsAsFactors = FALSE
  ) %>%
    mutate(
      OR_CI = paste0(OR, " (", CI_lower, "–", CI_upper, ")"),
      sig   = case_when(
        p_value < 0.001 ~ "***",
        p_value < 0.01  ~ "**",
        p_value < 0.05  ~ "*",
        TRUE            ~ ""
      )
    )
}

# ── Compile results per tab ───────────────────────────────────────────────────
compile_tab <- function(cpd_models, ces_models, group_label) {
  bind_rows(
    # Model 1
    extract_model_aim3(cpd_models$m1_EUR, "Model 1 (PRS + Demographics)", "EUR", "CPD", group_label),
    extract_model_aim3(cpd_models$m1_AFR, "Model 1 (PRS + Demographics)", "AFR", "CPD", group_label),
    extract_model_aim3(ces_models$m1_EUR, "Model 1 (PRS + Demographics)", "EUR", "CES", group_label),
    extract_model_aim3(ces_models$m1_AFR, "Model 1 (PRS + Demographics)", "AFR", "CES", group_label),
    # Model 2
    extract_model_aim3(cpd_models$m2_EUR, "Model 2 (+ Psychiatric)",      "EUR", "CPD", group_label),
    extract_model_aim3(cpd_models$m2_AFR, "Model 2 (+ Psychiatric)",      "AFR", "CPD", group_label),
    extract_model_aim3(ces_models$m2_EUR, "Model 2 (+ Psychiatric)",      "EUR", "CES", group_label),
    extract_model_aim3(ces_models$m2_AFR, "Model 2 (+ Psychiatric)",      "AFR", "CES", group_label),
    # Model 3
    extract_model_aim3(cpd_models$m3_EUR, "Model 3 (+ Psychiatric + SDoH)", "EUR", "CPD", group_label),
    extract_model_aim3(cpd_models$m3_AFR, "Model 3 (+ Psychiatric + SDoH)", "AFR", "CPD", group_label),
    extract_model_aim3(ces_models$m3_EUR, "Model 3 (+ Psychiatric + SDoH)", "EUR", "CES", group_label),
    extract_model_aim3(ces_models$m3_AFR, "Model 3 (+ Psychiatric + SDoH)", "AFR", "CES", group_label)
  )
}

results_GVE <- compile_tab(gve_CPD, gve_CES, "GVE")
results_SVE <- compile_tab(sve_CPD, sve_CES, "SVE")
results_LVE <- compile_tab(lve_CPD, lve_CES, "LVE")

# ── Write Excel ───────────────────────────────────────────────────────────────
library(openxlsx)

wb <- createWorkbook()

header_style <- createStyle(
  fontColour = "#FFFFFF", fgFill = "#2E4057",
  halign = "CENTER", textDecoration = "Bold"
)
sig_style <- createStyle(fgFill = "#FFF3CD")

write_tab <- function(wb, sheet_name, df) {
  addWorksheet(wb, sheet_name)
  writeData(wb, sheet_name, df)
  addStyle(wb, sheet_name, header_style,
           rows = 1, cols = 1:ncol(df), gridExpand = TRUE)
  sig_rows <- which(df$p_value < 0.05) + 1
  if (length(sig_rows) > 0) {
    addStyle(wb, sheet_name, sig_style,
             rows = sig_rows, cols = 1:ncol(df),
             gridExpand = TRUE, stack = TRUE)
  }
  setColWidths(wb, sheet_name, cols = 1:ncol(df), widths = "auto")
  freezePane(wb, sheet_name, firstRow = TRUE)
}

write_tab(wb, "GVE", results_GVE)
write_tab(wb, "SVE", results_SVE)
write_tab(wb, "LVE", results_LVE)

saveWorkbook(wb, "varenicline_effectiveness_results.xlsx", overwrite = TRUE)
cat("Saved: varenicline_effectiveness_results.xlsx\n")

## 6. Mediation of CPD<sup>PGS</sup> → cessation by cigarettes per day (Figure 3)

In [ ]:
# lavaan needs numeric covariates: sex coded Male = 1, Female = 0, other/skip = NA
prep_sem <- function(df) {
  df %>% mutate(sex = case_when(sex == "Male" ~ 1, sex == "Female" ~ 0, TRUE ~ NA_real_))
}
sem_data <- list(GVE = prep_sem(vare_all), SVE = prep_sem(sve_outcomes), LVE = prep_sem(lve_outcomes))

covs <- "age_at_treatment + sex +
         SCZ + ADHD + Bipolar_Disorder + Anxiety + Depression + PTSD +
         PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10"

# NOTE (cleanup): the original SVE cells fitted `model_multiple_lve` before it was
# defined. Both were the same single-mediator model, which is defined once here.
model_cpd <- paste0('
  Cigarettes_Per_Day_Numeric ~ a1 * PRS_CPD_scaled + ', covs, '

  quit_success ~ c * PRS_CPD_scaled + b1 * Cigarettes_Per_Day_Numeric + ', covs, '

  indirect_cpd   := a1 * b1
  total_indirect := indirect_cpd
  total          := c + total_indirect
  prop_mediated  := total_indirect / total
')

key_params <- function(fit, labels) {
  parameterEstimates(fit) %>%
    filter(label %in% labels) %>%
    select(label, est, se, pvalue, ci.lower, ci.upper)
}

mediation_cpd <- imap_dfr(sem_data, function(d, grp) {
  map_dfr(c("EUR", "AFR"), function(anc) {
    fit <- sem(model_cpd, data = d %>% filter(ancestry == anc), estimator = "ML")
    key_params(fit, c("a1", "b1", "c", "indirect_cpd", "total", "prop_mediated")) %>%
      mutate(group = grp, ancestry = anc, n = lavInspect(fit, "nobs"))
  })
})

mediation_cpd %>% mutate(across(where(is.numeric), ~ round(., 3)))
write_csv(mediation_cpd, "aim3_mediation_cpd.csv")

### Sensitivity: cigarettes per day, smoking years and starting age as joint mediators (Table S22)

In [ ]:
model_multiple <- paste0('
  Cigarettes_Per_Day_Numeric ~ a1 * PRS_CPD_scaled + ', covs, '
  Smoking_Years_Numeric      ~ a2 * PRS_CPD_scaled + ', covs, '
  Starting_Age_Numeric       ~ a3 * PRS_CPD_scaled + ', covs, '

  quit_success ~ c * PRS_CPD_scaled +
                 b1 * Cigarettes_Per_Day_Numeric +
                 b2 * Smoking_Years_Numeric +
                 b3 * Starting_Age_Numeric + ', covs, '

  # Allow mediators to correlate
  Cigarettes_Per_Day_Numeric ~~ Smoking_Years_Numeric
  Cigarettes_Per_Day_Numeric ~~ Starting_Age_Numeric
  Smoking_Years_Numeric ~~ Starting_Age_Numeric

  indirect_cpd   := a1 * b1
  indirect_years := a2 * b2
  indirect_age   := a3 * b3
  total_indirect := indirect_cpd + indirect_years + indirect_age
  total          := c + total_indirect
  prop_mediated  := total_indirect / total
  prop_cpd       := indirect_cpd / total
  prop_years     := indirect_years / total
  prop_age       := indirect_age / total
')

# Complete cases as in the original analysis (which also required DEPRIVATION_INDEX).
# NOTE (cleanup): the original LVE cell fitted the single-mediator model for AFR and
# had no EUR model; all six group × ancestry fits are run here.
mediation_multi <- imap_dfr(sem_data, function(d, grp) {
  map_dfr(c("EUR", "AFR"), function(anc) {
    dd <- d %>%
      filter(ancestry == anc) %>%
      drop_na(quit_success, PRS_CPD_scaled, Cigarettes_Per_Day_Numeric,
              Smoking_Years_Numeric, Starting_Age_Numeric, age_at_treatment, sex,
              SCZ, ADHD, Bipolar_Disorder, Anxiety, Depression, PTSD,
              PC1:PC10, DEPRIVATION_INDEX)
    fit <- sem(model_multiple, data = dd, estimator = "ML")
    key_params(fit, c("a1", "a2", "a3", "b1", "b2", "b3", "c",
                      "indirect_cpd", "indirect_years", "indirect_age",
                      "total_indirect", "total", "prop_mediated",
                      "prop_cpd", "prop_years", "prop_age")) %>%
      mutate(group = grp, ancestry = anc, n = nrow(dd))
  })
})

mediation_multi %>% mutate(across(where(is.numeric), ~ round(., 3)))
write_csv(mediation_multi, "aim3_mediation_multiple_S22.csv")